<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/13_ab_testing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap13) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap13-ab-testing.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 12](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/12_observabilidad_evaluacion.ipynb) · ➡️ [Cap. 14](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/14_equipo_investigacion.ipynb)

# Capítulo 13 — Mejorar sin romper

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Tenemos un prompt mejor… en teoría. ¿Lo cambiamos para todos los clientes un viernes por la tarde?

| | |
|---|---|
| **🧠 Componente agéntico** | Mejora continua y experimentación |
| **La necesidad** | Validar cambios de comportamiento con tráfico real y datos antes de promoverlos |
| **Lo que construimos** | Un segundo runtime (**NovaWeb**, el chat de los visitantes) detrás de un **Gateway HTTP**, dos **configuration bundles** (el prompt actual y uno más breve), evaluación online y un **A/B test** con tráfico real generado desde el notebook |
| **🏠 En local** | ☁️ No aplica: experimentar requiere tráfico real repartido entre variantes. |
| **☁️ Por qué AWS al salir a Internet** | En local, cambiar el prompt es redesplegar para todos. Los configuration bundles separan configuración y código, y el Gateway reparte el tráfico y mide con evaluación online antes de promover. |
| **En el ordenador sería…** | **Probar una actualización en unos pocos equipos** antes de instalarla en todos |
| **Requisitos del caso** | RNF-06 |
| **Prerrequisito · duración** | Capítulo 12 · 75 min (más ~15 min de espera de resultados) |
| **⚠️ Dónde se rompe la analogía** | Una actualización en pocos equipos se prueba sobre máquinas; aquí se prueba **sobre conversaciones reales con clientes**. El muestreo y la evaluación online son obligatorios. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=13)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(13)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(13)
# ponerse_al_dia(13)   # ← descomenta si llegas tarde o algo se rompió

## 0. ¿Diferente o mejor?

> ✋ **Predice antes de ejecutar.** Alguien ha probado un prompt nuevo tres veces a mano y «parece mejor». ¿Cuántas conversaciones **reales** harían falta para demostrarlo? ¿Diez? ¿Cien? ¿Mil? Apunta tu cifra.

> 🧭 **Antes de construir:** abre la teoría: [**Actualizar en pocos equipos**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap13-ab-testing.html). Decide cuánto tráfico mandas a cada variante y cuándo pararías el experimento. Vuelve cuando sepas responder a esta pregunta: **¿qué diferencia hay entre un prompt «diferente» y un prompt «mejor», y cuántas conversaciones hacen falta para distinguirlos?**

**En dos líneas:** **mejora continua** es cambiar el comportamiento (prompt, modelo, herramientas) con evidencia, no con opiniones: separar configuración y código para cambiar sin redesplegar, repartir tráfico real entre variantes y decidir con significancia estadística. En este capítulo lo haremos de verdad, en pequeño: dos variantes de prompt, tráfico generado desde el notebook y la evaluación online como juez.

**Hipótesis:** *«Respuestas más breves, que terminan con una pregunta de seguimiento, no empeoran la utilidad (`Builtin.Helpfulness`) del chat de visitantes»*.

## 1. NovaWeb: el chat de los visitantes

El 70 % de las conversaciones del chat de la web son de **visitantes sin sesión iniciada**: plazos, envíos, garantía. Es el sitio ideal para experimentar con el tono: no hay datos de clientes, ni pedidos, ni memoria.

NovaWeb usa **el mismo código y las mismas dependencias** que NovaAgent, con otro *entrypoint* (`web.py`): sin herramientas de pedidos, sin memoria, y con el *system prompt* que le llegue en el *configuration bundle* de cada petición (si no llega ninguno, usa el suyo).

In [ ]:
%%writefile NovaAssist/app/NovaAgent/web.py
"""NovaWeb: el chat de la web para visitantes sin sesión iniciada (capítulo 13).

Mismo código y dependencias que NovaAgent, otro entrypoint. No tiene herramientas de pedidos ni memoria: no sabe
quién es el visitante. Su system prompt puede llegar en un configuration bundle (A/B test del Gateway).
"""
from bedrock_agentcore.runtime import BedrockAgentCoreApp, BedrockAgentCoreContext
from strands import Agent

from modelo import crear_modelo
from respuesta import respuesta

app = BedrockAgentCoreApp()

PROMPT_VISITANTES = """Eres Nova Assist, el asistente de la web de TiendaNova para visitantes que no han iniciado sesión.
Te presentas como asistente de inteligencia artificial. No tienes acceso a pedidos: si preguntan por uno, explica
que deben iniciar sesión. Datos oficiales de TiendaNova que puedes usar:
- Devoluciones: 30 días naturales desde la entrega (el mínimo legal en la UE es 14; TiendaNova lo amplía).
  Gratuitas si el producto es defectuoso; en el resto de casos se descuentan 4,95 €. Los auriculares in-ear
  desprecintados no se admiten por higiene.
- Reembolsos: al método de pago original, en un máximo de 14 días desde que se recibe y revisa el producto.
- Garantía: 3 años desde la entrega; no cubre mal uso, golpes, líquidos ni desgaste de baterías.
- Envíos: 24-72 horas laborables en la península; 3-5 días laborables en Baleares y Canarias.
- Equipo humano: de lunes a viernes de 9:00 a 20:00 y sábados de 10:00 a 14:00.
Si no sabes algo, dilo y ofrece el contacto con el equipo humano."""


def prompt_del_bundle() -> str:
    """System prompt del configuration bundle de esta petición; si no llega ninguno, el de por defecto."""
    try:
        return BedrockAgentCoreContext.get_config_bundle().get("system_prompt") or PROMPT_VISITANTES
    except Exception:  # noqa: BLE001  (servicio de bundles no disponible: se degrada al prompt por defecto)
        return PROMPT_VISITANTES


def variante() -> str | None:
    """Variante del A/B que atiende la petición (la marca el Gateway) o, en su defecto, el bundle recibido."""
    ref = BedrockAgentCoreContext.get_config_bundle_ref()
    return BedrockAgentCoreContext.get_routing_experiment_variant() or (ref.bundle_id if ref else None)


@app.entrypoint
def invoke(payload, context):
    agent = Agent(model=crear_modelo(), system_prompt=prompt_del_bundle(), callback_handler=None)
    return respuesta(agent(payload.get("prompt", "")), variante=variante())


if __name__ == "__main__":
    app.run()

In [ ]:
cli_args("add", "agent", "--name", "NovaWeb", "--type", "byo", "--build", "CodeZip", "--language", "Python",
         "--framework", "Strands", "--model-provider", "Bedrock",
         "--code-location", "app/NovaAgent/", "--entrypoint", "web.py")
if state_get("GUARDRAIL_ID"):   # web.py usa el mismo modelo.py: necesita el mismo permiso sobre el guardrail
    anadir_permiso_runtime("guardrail", [{
        "Effect": "Allow", "Action": ["bedrock:ApplyGuardrail"],
        "Resource": f"arn:aws:bedrock:{region()}:{state_get('ACCOUNT_ID')}:guardrail/{state_get('GUARDRAIL_ID')}"}],
        runtime="NovaWeb")

## 2. El Gateway del experimento

El reparto del tráfico lo hace un **Gateway HTTP** (protocolo `None`, no MCP) con un *target* que apunta al runtime NovaWeb. La web llamaría a `https://<gateway>/web/invocations` firmando con IAM; el notebook hará lo mismo.

In [ ]:
cli_args("add", "gateway", "--name", "NovaWebGateway", "--protocol-type", "None", "--authorizer-type", "AWS_IAM")
cli_args("add", "gateway-target", "--name", "web", "--type", "http-runtime", "--runtime", "NovaWeb",
         "--gateway", "NovaWebGateway")

## 3. Las dos variantes: *configuration bundles*

Un *configuration bundle* es configuración versionada **fuera del código**: aquí, el *system prompt* de NovaWeb. Cambiarlo no exige redesplegar el agente. Creamos dos: el prompt actual (control) y el breve (tratamiento).

In [ ]:
import json, sys
sys.path.insert(0, str(AGENT_DIR))
from web import PROMPT_VISITANTES

PROMPT_BREVE = PROMPT_VISITANTES + """
Formato: responde en un máximo de dos frases y termina con una pregunta breve que ayude al visitante a dar el
siguiente paso."""

for nombre, descripcion, prompt in (("NovaWebActual", "Prompt actual del chat de visitantes", PROMPT_VISITANTES),
                                    ("NovaWebBreve", "Respuestas breves con pregunta de seguimiento", PROMPT_BREVE)):
    cli_args("add", "config-bundle", "--name", nombre, "--description", descripcion,
             "--components", json.dumps({"{{runtime:NovaWeb}}": {"configuration": {"system_prompt": prompt}}},
                                        ensure_ascii=False))

## 4. El juez: evaluación online de NovaWeb

El A/B necesita una métrica. Usamos `Builtin.Helpfulness` sobre el **100 %** de las sesiones de NovaWeb: en un experimento pequeño cada conversación cuenta (en producción, con miles de sesiones, se muestrea).

In [ ]:
cli_args("add", "online-eval", "--name", "nova_web_eval", "--runtime", "NovaWeb",
         "--evaluator", "Builtin.Helpfulness", "--sampling-rate", "100", "--enable-on-create")
assert cli("agentcore validate") == 0
cli("agentcore deploy --yes")

In [ ]:
gw_web = buscar_gateway_http("NovaWebGateway")
URL_WEB = f"{gw_web['url_base']}/web/invocations"
state_set(AB_GATEWAY_URL=URL_WEB)
invocar("Hola, ¿cuánto tarda un envío a Canarias?", url=URL_WEB)   # sin A/B todavía: prompt por defecto

## 5. Lanzar el A/B

El Gateway reparte las **sesiones** (no los mensajes) al 50 %: una sesión se queda siempre en la misma variante. A cada petición le añade la referencia del *bundle* de su variante; el SDK la lee y `web.py` usa ese prompt.

In [ ]:
cli_args("run", "ab-test", "--name", "tono_breve", "--gateway", "NovaWebGateway", "--runtime", "NovaWeb",
         "--mode", "config-bundle",
         "--control-bundle", "NovaWebActual", "--control-version", "LATEST",
         "--treatment-bundle", "NovaWebBreve", "--treatment-version", "LATEST",
         "--online-eval", "nova_web_eval", "--control-weight", "50", "--treatment-weight", "50")

import boto3
dp = boto3.client("bedrock-agentcore", region_name=region())
ab = next(t for t in dp.list_ab_tests()["abTests"] if t["name"] == "tono_breve")
state_set(AB_TEST_ID=ab["abTestId"])
print(ab["status"], ab["executionStatus"])

## 6. Tráfico

Cuarenta visitantes, una pregunta cada uno, en serie (para no rozar las cuotas del modelo). Fíjate en la línea 🧪.

> ✋ **Predice antes de ejecutar.** ¿Saldrá un reparto exacto de 20 y 20? ¿Se notará la diferencia entre variantes a simple vista?

In [ ]:
import random, statistics
PREGUNTAS = ["¿Cuántos días tengo para devolver un producto?", "¿Cuánto tarda un envío a Mallorca?",
             "¿Qué garantía tienen los productos?", "¿Puedo devolver unos auriculares in-ear abiertos?",
             "¿Cuándo me devolvéis el dinero de una devolución?", "¿Cuesta algo devolver un producto que no me gusta?",
             "¿A qué hora atiende el equipo humano?", "Me ha llegado un producto roto, ¿qué hago?",
             "¿Dónde está mi pedido TN-1001?", "¿La garantía cubre la batería?"]
random.seed(13)
por_variante = {}
sesiones_web = []
for i in range(40):
    sid = invocar(random.choice(PREGUNTAS), url=URL_WEB)
    sesiones_web.append(sid)
    por_variante.setdefault(ultimo_dato("variante") or "sin variante", []).append(len((ultima_respuesta() or "").split()))
for variante, longitudes in por_variante.items():
    print(f"{variante:12} sesiones={len(longitudes):3}  palabras por respuesta (mediana)={statistics.median(longitudes):.0f}")

El reparto no será exacto (es aleatorio por sesión) y la diferencia de **longitud** se ve enseguida: eso es «diferente», y se mide con una regla. Si es **mejor** lo dirá el juez, sesión a sesión.

In [ ]:
# Cerramos las microVM de las sesiones: si no, siguen vivas hasta 15 minutos sin actividad
ctl = boto3.client("bedrock-agentcore-control", region_name=region())
arn_web = next(r["agentRuntimeArn"] for r in ctl.list_agent_runtimes()["agentRuntimes"] if "NovaWeb" in r["agentRuntimeName"])
cerradas = 0
for sid in sesiones_web:
    try:
        dp.stop_runtime_session(agentRuntimeArn=arn_web, runtimeSessionId=sid)
        cerradas += 1
    except Exception:  # noqa: BLE001  (la sesión ya había terminado)
        pass
print(f"{cerradas} de {len(sesiones_web)} sesiones cerradas")

## 7. Resultados

La evaluación online puntúa las sesiones cuando terminan y el análisis del A/B tarda **unos 15 minutos** en aparecer: buen momento para la pausa. Después, ejecuta la celda siguiente (puedes repetirla).

In [ ]:
r = dp.get_ab_test(abTestId=state_get("AB_TEST_ID"))
print(r["status"], r["executionStatus"])
for m in (r.get("results") or {}).get("evaluatorMetrics", []):
    c = m["controlStats"]
    print(f"{m['evaluatorArn'].rsplit('/', 1)[-1]}: control n={c['sampleSize']} media={c['mean']:.3f}")
    for v in m["variantResults"]:
        print(f"   {v['variantName']}: n={v['sampleSize']} media={v['mean']:.3f} cambio={v['percentChange']:+.1f} % "
              f"p={v['pValue']:.3f} significativo={v['isSignificant']}")
if not (r.get("results") or {}).get("evaluatorMetrics"):
    print("Todavía sin resultados: espera unos minutos y repite la celda.")

## 8. Decidir

La regla se fija **antes** de mirar: promovemos el tratamiento solo si la diferencia es significativa y no empeora la utilidad. Con 40 sesiones lo normal es que **no** sea significativa: el experimento no demuestra nada todavía, y eso también es un resultado. En producción, el test seguiría días hasta reunir las sesiones necesarias.

In [ ]:
resultados = (dp.get_ab_test(abTestId=state_get("AB_TEST_ID")).get("results") or {}).get("evaluatorMetrics", [])
tratamiento = [v for m in resultados for v in m["variantResults"]]
if tratamiento and all(v["isSignificant"] and v["absoluteChange"] >= 0 for v in tratamiento):
    print("✔ El tratamiento gana: lo promovemos (detiene el test y lo escribe en agentcore.json)")
    cli_args("promote", "ab-test", "-i", state_get("AB_TEST_ID"))
    cli("agentcore deploy --yes")
else:
    print("= Sin evidencia suficiente: detenemos el test y NovaWeb sigue con el prompt actual")
    cli_args("stop", "ab-test", "-i", state_get("AB_TEST_ID"))

> **Una sola prueba A/B en marcha por Gateway.** Si quieres repetir el experimento con otra hipótesis, detén antes el anterior. El capítulo 17 borra el test, los *bundles*, NovaWeb y su Gateway.

## 9. Recommendations (opcional)

Pide **Recommendations** sobre las trazas de producción para que AgentCore proponga mejoras del *system prompt* y de las descripciones de herramientas (`agentcore run recommendation --help`). Valida la propuesta con ***batch evaluations*** sobre el conjunto de prueba del capítulo 12 antes de lanzar otro A/B (lo haremos en el capítulo 15).

## Para pensar

- ¿Qué cambios merecen un A/B y cuáles basta con validar con *batch evaluations*?
- ¿Por qué el Gateway reparte **sesiones** y no mensajes?
- ¿Qué otra métrica, además de la utilidad, vigilarías en este experimento (coste, longitud, escalados)?

## 🔎 Lo que acabamos de descubrir

Dirección insiste: *«¿Por qué se devuelven tantos robots S9? ¿Es cosa nuestra o del sector? ¿Qué hacen los demás? Quiero un informe el lunes.»* No es una conversación con un cliente: es una **investigación**, con tres trabajos distintos.

In [ ]:
cuaderno()
taximetro_cerrar(13)
guardar_partida(13)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap13-ab-testing.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap13-ab-testing.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `14_equipo_investigacion.ipynb` — *Del agente al equipo*

---
**Siguiente:** ➡️ [Capítulo 14 · Del agente al equipo](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/14_equipo_investigacion.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap14). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.